In [ ]:
import json
from typing import List, Dict
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

from scipy.stats import pearsonr
from tqdm import tqdm
import math
import re
import requests

from torch.optim.lr_scheduler import ReduceLROnPlateau
import wandb
from datetime import datetime
from transformers import get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup
import os
from google.colab import drive
from google.colab import userdata

userdata.get('HF_TOKEN')

drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_jsonl(filepath: str) -> List[Dict]:
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def load_jsonl_url(url: str) -> List[Dict]:
    resp = requests.get(url)
    resp.raise_for_status()
    return [json.loads(line) for line in resp.text.splitlines()]

Mounted at /content/drive


In [ ]:
#==== step 1 load the data ====
# you can change the env for your task.
# train data should have the VA labels, predit data without VA labels

def jsonl_to_df(data):
    if 'Quadruplet' in data[0]:
        df = pd.json_normalize(data, 'Quadruplet', ['ID', 'Text'])
        df[['Valence', 'Arousal']] = df['VA'].str.split('#', expand=True).astype(float)
        df = df.drop(columns=['VA', 'Category', 'Opinion'])  # drop unnecessary columns
        df = df.drop_duplicates(subset=['ID', 'Aspect'], keep='first')  # remove duplicate ID+Aspect

    elif 'Aspect' in data[0]:
        df = pd.json_normalize(data, 'Aspect', ['ID', 'Text'])
        df = df.rename(columns={df.columns[0]: "Aspect"})  # rename to Aspect
        df['Valence'] = 0  # default value
        df['Arousal'] = 0  # default value

    elif 'Aspect_VA' in data[0]:
        df = pd.json_normalize(data, 'Aspect_VA', ['ID', 'Text'])
        df[['Valence', 'Arousal']] = df['VA'].str.split('#', expand=True).astype(float)
        df = df.drop_duplicates(subset=['ID', 'Aspect'], keep='first')  # remove duplicate ID+Aspect

    else:
        raise ValueError("Invalid format: must include 'Quadruplet' or 'Aspect'")

    return df

In [ ]:
!pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 27.0 MB/s eta 0:00:00


In [ ]:
#==== step 4 use dev data to check your model's performance ====
def get_prd(model,dataloder, type ="dev"):
    if type == "dev":
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in dataloder:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].cpu().numpy()
                outputs, emb = model(input_ids, attention_mask)
                outputs = outputs.clamp(-1.0, 1.0)
                outputs = outputs.cpu().numpy()
                all_preds.append(outputs)
                all_labels.append(labels)
        preds = np.vstack(all_preds)
        lables = np.vstack(all_labels)

        pred_v = preds[:, 0] * 4.0 + 5.0
        pred_a = preds[:, 1] * 4.0 + 5.0

        gold_v = lables[:,0] * 4.0 + 5.0
        gold_a = lables[:,1] * 4.0 + 5.0

        return pred_v, pred_a, gold_v, gold_a

    elif type == "pred":
        all_preds = []
        with torch.no_grad():
            for batch in dataloder:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                outputs, emb  = model(input_ids, attention_mask)
                outputs = outputs.clamp(-1.0, 1.0)
                outputs = outputs.cpu().numpy()
                all_preds.append(outputs)
        preds = np.vstack(all_preds)

        pred_v = preds[:, 0] * 4.0 + 5.0
        pred_a = preds[:, 1] * 4.0 + 5.0

        return pred_v, pred_a

def evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v):
    if not (all(1 <= x <= 9 for x in pred_v) and all(1 <= x <= 9 for x in pred_a)):
        print(f"Warning: Some predicted values are out of the numerical range.")
    pcc_v = pearsonr(pred_v, gold_v)[0]
    pcc_a = pearsonr(pred_a, gold_a)[0]

    gold_va = gold_v + gold_a
    pred_va = pred_v + pred_a

    def rmse(gold_va, pred_va):
        result = [(a - b) ** 2 for a, b in zip(gold_va, pred_va)]
        return math.sqrt(sum(result)/len(gold_v))

    rmse_va = rmse(gold_va, pred_va)
    return {
        'PCC_V': pcc_v,
        'PCC_A': pcc_a,
        'RMSE_VA': rmse_va,
    }

In [ ]:
class VADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.sentences = dataframe["Text"].tolist()
        self.aspects = dataframe["Aspect"].tolist()
        # Normalize valence/arousal to [-1, 1]
        self.labels = ((dataframe[["Valence", "Arousal"]].values.astype(float) - 5.0) / 4.0)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        text = f"Aspect: {self.aspects[idx]}. Sentence: {self.sentences[idx]}."
        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"][0]

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from tqdm import tqdm

def concordance_cc(preds, targets):
    pred_mean = preds.mean(dim=0)
    true_mean = targets.mean(dim=0)
    cov = ((preds - pred_mean) * (targets - true_mean)).mean(dim=0)

    # Use unbiased=False to avoid DoF warning for small batches
    pred_var = preds.var(dim=0, unbiased=False) + 1e-8
    true_var = targets.var(dim=0, unbiased=False) + 1e-8

    ccc = (2 * cov) / (pred_var + true_var + (pred_mean - true_mean)**2)
    return ccc

def ccc_mse_loss(preds, targets, mse_weight=0.4, valence_weight=0.3, arousal_weight=0.7):
    mse = torch.nn.functional.mse_loss(preds, targets)
    ccc = concordance_cc(preds, targets)
    ccc_mean = valence_weight * ccc[0] + arousal_weight * ccc[1]
    return mse_weight * mse + (1 - mse_weight) * (1 - ccc_mean)


# ==== Model ====
class TransformerVARegressor(nn.Module):
    def __init__(self, model_name, dropout=0.3, add_label_noise=True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size

        # Two separate heads (helps PCC_A)
        self.dropout = nn.Dropout(dropout)
        self.reg_head_v = nn.Linear(hidden_size, 1)
        self.reg_head_a = nn.Linear(hidden_size, 1)

        self.attention = nn.Linear(hidden_size, 1)

        self.add_label_noise = add_label_noise

    def forward(self, input_ids, attention_mask):
      """
      aspect_mask: tensor of shape [batch, seq_len], 1 for aspect tokens, 0 elsewhere
      """
      # Standard backbone call
      outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
      hidden = outputs.last_hidden_state  # [batch, seq_len, hidden_size]

      # Compute attention scores
      scores = self.attention(hidden).squeeze(-1)  # [batch, seq_len]

      # Mask padding tokens
      scores = scores.masked_fill(attention_mask == 0, -1e9)

      # Softmax weights and pooling
      weights = torch.softmax(scores, dim=1).unsqueeze(-1)  # [batch, seq_len, 1]
      mean_pooled = torch.sum(hidden * weights, dim=1)  # [batch, hidden_size]

      mean_pooled = self.dropout(mean_pooled)

      # Separate regression heads
      v = self.reg_head_v(mean_pooled)
      a = self.reg_head_a(mean_pooled)

      output = torch.cat([v, a], dim=1)

      return output, mean_pooled


    # ==== Training ====
    def train_epoch(self, loader, optimizer, device, scheduler, loss_fn, max_grad_norm=1.0, warmup_steps=None):
      self.train()
      total_loss = 0.0

      for batch in tqdm(loader, desc="Training", leave=False):
          optimizer.zero_grad()

          input_ids = batch["input_ids"].to(device)
          attention_mask = batch["attention_mask"].to(device)
          labels = batch["labels"].to(device)

          # FORWARD PASS (NOW RETURNS PRED & EMBEDDING)
          outputs, emb = self(input_ids, attention_mask)
          loss = loss_fn(outputs, labels,emb)
          loss.backward()

          torch.nn.utils.clip_grad_norm_(self.parameters(), max_grad_norm)
          optimizer.step()

          # Step warm-up scheduler only during first warmup_steps

          scheduler.step()

          total_loss += loss.item()

      avg_loss = total_loss / len(loader)
      return avg_loss

    # ==== Evaluation ====
    def eval_epoch(self, loader, device, loss_fn):
        self.eval()
        total_loss = 0
        with torch.no_grad():
            for batch in tqdm(loader, desc="Evaluating", leave=False):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs,emb = self(input_ids, attention_mask)
                outputs = outputs.clamp(-1.0, 1.0)
                loss = loss_fn(outputs, labels,emb)
                total_loss += loss.item()

        return total_loss / len(loader)

In [ ]:
def va_distance(v1, a1, v2, a2):
    """
    Computes Euclidean distance in normalized VA space.
    Inputs are assumed in [-1, 1] (normalized).
    """
    return torch.sqrt((v1 - v2)**2 + (a1 - a2)**2 + 1e-8)

def sample_triplets(labels, positive_threshold=0.3, negative_threshold=0.9):
    """
    labels: tensor [B, 2] of normalized VA labels
    Returns indices: anchors, positives, negatives
    """

    B = labels.size(0)
    V = labels[:, 0]
    A = labels[:, 1]

    anchors, positives, negatives = [], [], []

    for i in range(B):
        # compute distance from anchor i to all other samples
        dists = va_distance(V[i], A[i], V, A)

        # positive = close in VA space
        pos_candidates = torch.where(dists < positive_threshold)[0]
        pos_candidates = pos_candidates[pos_candidates != i]  # remove self

        # negative = far in VA space
        neg_candidates = torch.where(dists > negative_threshold)[0]

        if len(pos_candidates) == 0 or len(neg_candidates) == 0:
            continue

        # random pick
        p = pos_candidates[torch.randint(0, len(pos_candidates), (1,))]
        n = neg_candidates[torch.randint(0, len(neg_candidates), (1,))]

        anchors.append(i)
        positives.append(p.item())
        negatives.append(n.item())

    if len(anchors) == 0:
        # fallback: no valid triplets found
        return None, None, None

    return (
        torch.tensor(anchors),
        torch.tensor(positives),
        torch.tensor(negatives),
    )


class ContrastiveVALoss(nn.Module):
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):
        """
        embeddings: [B, H]
        labels: normalized VA labels [B, 2]
        """

        anchors, positives, negatives = sample_triplets(labels)

        if anchors is None:
            return torch.tensor(0.0, device=embeddings.device)

        # gather embeddings
        E = embeddings
        A = E[anchors]
        P = E[positives]
        N = E[negatives]

        # compute distances
        d_ap = torch.norm(A - P, dim=1)
        d_an = torch.norm(A - N, dim=1)

        # triplet loss
        loss = torch.relu(d_ap - d_an + self.margin)

        return loss.mean()

In [ ]:
config = {
    'model_name': "DeepPavlov/rubert-base-cased",
    'rs': 42,
    'lr': 2e-5,
    'batch_size': 32,
    'dropout': 0.3,
    'warmup_ratio': 0.05,
    'early_stopping_patience': 5,
    'max_len': 128,
    'scheduler': ["Warmup + ReduceLROnPlateau"],
    'optimizer': ["AdamW"],

    # Fixed parameters
    'subtask': "subtask_1",
    'task': "task1",
    'language': "rus",
    'domain': "restaurant",
    'epochs': 30,  # small model converges faster
    'loss_fn': "CCC + MSELoss",
    'metrics': ["RMSE_VA", "PCC_V", "PCC_A"],
    'save_dir': "/content/drive/MyDrive/DimABSA2026/final_models",
    'url' : f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a",
}

In [ ]:
wandb.init(
    project="DimABSA2026-Subtask1",
    name=f"{config['model_name'].replace('/', '_')}_{config['language']}_{config['domain']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    config=config
)

config = wandb.config

save_path = os.path.join(
  config.save_dir,
  config.language,
  config.domain,
  config.model_name.replace("/", "_"),
  wandb.run.id  # unique ID per sweep run
)
os.makedirs(save_path, exist_ok=True)

if config.domain == 'finance':
  train_url = config.url + f'/{config.subtask}/{config.language}/{config.language}_{config.domain}_train_{config.task}.jsonl'
else:
  train_url = config.url + f'/{config.subtask}/{config.language}/{config.language}_{config.domain}_train_alltasks.jsonl'

predict_url = config.url + f'/{config.subtask}/{config.language}/{config.language}_{config.domain}_dev_{config.task}.jsonl'

train_raw = load_jsonl_url(train_url)
predict_raw = load_jsonl_url(predict_url)

train_df = jsonl_to_df(train_raw)
predict_df = jsonl_to_df(predict_raw)

# split 10% for dev
train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=config.rs)

# convert to Dataset and Dataloader
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

train_dataset = VADataset(train_df, tokenizer, max_len=config.max_len)
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)

dev_dataset = VADataset(dev_df, tokenizer, max_len=config.max_len)
dev_loader = DataLoader(dev_dataset, batch_size=config.batch_size, shuffle=False)

model = TransformerVARegressor(config.model_name,dropout=config.dropout).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)

contrastive_va = ContrastiveVALoss(margin=0.5)

def combined_va_loss(pred, gold, embeddings):
    # your existing loss
    base_loss = ccc_mse_loss(
        pred, gold,
        mse_weight=0.4,
        valence_weight=0.3,
        arousal_weight=0.7
    )

    embeddings = embeddings.detach()

    # contrastive triplet loss
    contrast_loss = contrastive_va(embeddings, gold)

    # combine them
    loss = (
        0.95 * base_loss +     # this keeps your core behavior
        0.05 * contrast_loss   # gentle push to structure VA space
    )

    return loss

loss_fn = combined_va_loss

total_steps = len(train_loader) * config.epochs
warmup_steps = int(total_steps * config.warmup_ratio)

scheduler_warmup = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scheduler_plateau = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

# Initialize best_val_loss and patience_counter before the training loop
best_rmse_va = float("inf")     # <-- ADD THIS
patience_counter = 0


for epoch in range(config.epochs):
    train_loss = model.train_epoch(train_loader, optimizer, device, scheduler_warmup, loss_fn, warmup_steps=warmup_steps)

    val_loss = model.eval_epoch(dev_loader, device, loss_fn)

    # ==== Early Stopping + Model Saving (based on mean CCC) ====
    pred_v, pred_a, gold_v, gold_a = get_prd(model, dev_loader, type="dev")

    eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
    current_rmse = eval_score["RMSE_VA"]

    ccc_v = concordance_cc(torch.tensor(pred_v), torch.tensor(gold_v))
    ccc_a = concordance_cc(torch.tensor(pred_a), torch.tensor(gold_a))


    scheduler_plateau.step(current_rmse)

    # Log to W&B
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "RMSE_VA": float(current_rmse),  # <-- Log your main metric
        "CCC_V": float(ccc_v),
        "CCC_A": float(ccc_a),
        "lr": optimizer.param_groups[0]['lr'],
    })

    # Print summary
    print(f"model:{config.model_name} | Epoch:{epoch+1} | "
          f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
          f"RMSE_VA={current_rmse:.4f}, CCC_V={ccc_v:.4f}, CCC_A={ccc_a:.4f}")

    if current_rmse < best_rmse_va:  # Lower is better
        best_rmse_va = current_rmse
        patience_counter = 0

        best_path = os.path.join(save_path, f"best_model.pt")
        torch.save(model.state_dict(), best_path)

        wandb.run.summary["best_RMSE_VA"] = best_rmse_va # <-- Update summary
        wandb.run.summary["best_model_path"] = best_path
        print(f"New best model saved at epoch {epoch+1}, at {best_path}")
    else:
        patience_counter += 1
        if patience_counter >= config.early_stopping_patience:
            print("Early stopping triggered.")
            break

model.load_state_dict(torch.load(best_path))
model.eval()

# lap dev score
pred_v, pred_a, gold_v, gold_a = get_prd(model, dev_loader,type="dev")
eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)

wandb.log({
    "PCC_V": float(eval_score["PCC_V"]),
    "PCC_A": float(eval_score["PCC_A"]),
    "RMSE_VA": float(eval_score["RMSE_VA"]),
})

print(f"{config.model_name} dev_eval: {eval_score}")
wandb.finish()

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


model:DeepPavlov/rubert-base-cased | Epoch:1 | train_loss=0.6037, val_loss=0.4324, RMSE_VA=1.9020, CCC_V=0.5583, CCC_A=0.4167
New best model saved at epoch 1, at /content/drive/MyDrive/DimABSA2026/final_models/rus/restaurant/DeepPavlov_rubert-base-cased/gk88lph4/best_model.pt


model:DeepPavlov/rubert-base-cased | Epoch:2 | train_loss=0.3861, val_loss=0.3855, RMSE_VA=2.1485, CCC_V=0.6962, CCC_A=0.3861


model:DeepPavlov/rubert-base-cased | Epoch:3 | train_loss=0.2582, val_loss=0.3264, RMSE_VA=1.7272, CCC_V=0.7157, CCC_A=0.5233
New best model saved at epoch 3, at /content/drive/MyDrive/DimABSA2026/final_models/rus/restaurant/DeepPavlov_rubert-base-cased/gk88lph4/best_model.pt


model:DeepPavlov/rubert-base-cased | Epoch:4 | train_loss=0.1946, val_loss=0.2907, RMSE_VA=1.6805, CCC_V=0.7894, CCC_A=0.5841
New best model saved at epoch 4, at /content/drive/MyDrive/DimABSA2026/final_models/rus/restaurant/DeepPavlov_rubert-base-cased/gk88lph4/best_model.pt


...

model:DeepPavlov/rubert-base-cased | Epoch:27 | train_loss=0.0174, val_loss=0.2320, RMSE_VA=1.3673, CCC_V=0.8291, CCC_A=0.6993


model:DeepPavlov/rubert-base-cased | Epoch:28 | train_loss=0.0173, val_loss=0.2360, RMSE_VA=1.3592, CCC_V=0.8330, CCC_A=0.6897


model:DeepPavlov/rubert-base-cased | Epoch:29 | train_loss=0.0180, val_loss=0.2318, RMSE_VA=1.3563, CCC_V=0.8368, CCC_A=0.7010


model:DeepPavlov/rubert-base-cased | Epoch:30 | train_loss=0.0160, val_loss=0.2302, RMSE_VA=1.3574, CCC_V=0.8329, CCC_A=0.7011
Early stopping triggered.
DeepPavlov/rubert-base-cased dev_eval: {'PCC_V': np.float32(0.8443145), 'PCC_A': np.float32(0.702313), 'RMSE_VA': 1.339971686391474}


CCC_A,▂▁▄▅▅▅▆▇▇▇▇▇▇███▇██████▇██████
CCC_V,▁▄▅▇▆▇▆▇▇▇▇███████████████████
PCC_A,▁
PCC_V,▁
RMSE_VA,▆█▄▄▅▃▄▂▃▃▂▂▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,▆██▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▂▃▃▃▃▂▂▂▂▁▁▁
train_loss,█▅▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▆▄▃▃▃▃▂▂▂▂▁▂▁▁▁▂▁▁▁▁▁▁▂▁▁▁▁▁▁
CCC_A,0.70108
CCC_V,0.83292


In [ ]:
#==== step 5 save & submit your predict results ====
def extract_num(s):
    m = re.search(r"(\d+)$", str(s))
    return int(m.group(1)) if m else -1

def df_to_jsonl(df, out_path):
    df_sorted = df.sort_values(by="ID", key=lambda x: x.map(extract_num))
    grouped = df_sorted.groupby("ID", sort=False)

    with open(out_path, "w", encoding="utf-8") as f:
        for gid, gdf in grouped:
            record = {
                "ID": gid,
                "Aspect_VA": []
            }
            for _, row in gdf.iterrows():
                record["Aspect_VA"].append({
                    "Aspect": row["Aspect"],
                    "VA": f"{row['Valence']:.2f}#{row['Arousal']:.2f}"
                })
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

pred_dataset = VADataset(predict_df, tokenizer)
pred_loader = DataLoader(pred_dataset, batch_size=32, shuffle=False)
pred_v, pred_a, = get_prd(model, pred_loader,type="pred")

predict_df["Valence"] = pred_v
predict_df["Arousal"] = pred_a

df_to_jsonl(predict_df, f"pred_{config.language}_{config.domain}.jsonl")